# On-time performance & headway adherence — operations metrics, honestly labeled

> Explore and compute freely: nothing computed outside Headway's calculation library (services/calc) can ever become a reported figure. Only the calculation library writes computed.metric_values, and the walls are structural database CHECKs, not policy.

Headway also measures its namesake: how the service actually ran. On-time
performance (OTP) and headway adherence are **operations metrics** —
`category='ops'` — and the platform is structural about what that means:
they are never certifiable (a certified ops figure is impossible to even
represent in the database — migration 0024's CHECK), they never appear in
NTD packages or the public certified feed, and every surface that shows one
labels it. This notebook reads them and shows the refusal accounting that
comes with them.

**You need:** `HEADWAY_API_URL` and `HEADWAY_MACHINE_KEY` (scope
`read:metrics`) — see [docs/analyst-access.md](../docs/analyst-access.md).

*This notebook was executed once against a live Headway demo stack (2026-07-15) so its outputs are real; some demo figures come from simulated feeds and say so in their provenance. To run it yourself you need a running Headway API and the environment variables named below — credentials are never written into a notebook.*

In [1]:
import os

import pandas as pd

from headway_client import HeadwayClient, frames

pd.set_option("display.width", 160)

hw = HeadwayClient(
    os.environ.get("HEADWAY_API_URL", "http://127.0.0.1:8000"),
    token=os.environ["HEADWAY_MACHINE_KEY"],
)

## Agency-level operations figures

`category="ops"` filters on the honesty boundary. Note the provenance
columns: every row says `category='ops'` about itself, so no downstream
tool can mistake an operations number for a regulatory one.

In [2]:
ops = frames.metric_values_frame(hw.metric_values(category="ops"))
assert (ops["category"] == "ops").all()
print(f"{len(ops)} operations figures "
      f"({ops['metric'].value_counts().to_dict()})")
ops[ops["scope"] == "agency"].drop(
    columns=["detail", "source_mix", "computed_at"]
)

345 operations figures ({'otp': 173, 'headway_adherence': 172})


,metric,period_start,period_end,scope,value,unit,metric_value_id,calc_name,calc_version,category,certification_status,simulated
0,headway_adherence,2026-07-01,2026-08-01,agency,0.3010,ratio,9fb58ad8-cc5a-430a-9977-27a341b7669b,headway_adherence_v0,0.1.0,ops,uncertified,False
344,otp,2026-07-01,2026-08-01,agency,54.10,percent,5b43f678-4fe4-4496-ba22-4ff330be7be9,otp_v0,0.1.0,ops,uncertified,False


OTP here is the share of observed stop passages inside the
agency's configured on-time window; headway adherence is the coefficient
of variation of observed vs scheduled headways (lower is steadier). Their
definitions, bases, and deliberate limits are quoted and cited in
[`services/calc/OPS_DEFINITIONS.md`](../services/calc/OPS_DEFINITIONS.md) —
the operations analogue of the regulatory tracker.

## Refusal accounting, surfaced honestly

These metrics are *derived from* GPS positions matched to scheduled stops,
and the derivation refuses what it cannot defend: a passage whose position
cadence had gaps, a stop the vehicle never demonstrably reached, an
endpoint it cannot bound. Refusals are **counted, not hidden** — they ride
in every figure's detail, so you always know how much observation the
number stands on.

In [3]:
agency_otp = ops[(ops["metric"] == "otp") & (ops["scope"] == "agency")].iloc[0]
d = agency_otp["detail"]
derivation = d["derivation"]

print(f"OTP {agency_otp['value']} (window: -{d['early_tolerance_seconds']}s "
      f"/ +{d['late_tolerance_seconds']}s, timezone {d['agency_timezone']})")
print(f"on_time / early / late: {d['on_time_count']} / {d['early_count']} "
      f"/ {d['late_count']}  of {d['passages_considered']} passages")

pd.DataFrame(
    [
        ("positions considered", derivation["positions_considered"]),
        ("passages derived", derivation["passages_derived"]),
        ("REFUSED: cadence gap", derivation["refused_cadence_gap"]),
        ("REFUSED: stop not demonstrably reached",
         derivation["refused_not_reached"]),
        ("REFUSED: endpoint unbounded",
         derivation["refused_endpoint_unbounded"]),
        ("occurrences skipped (too few positions)",
         derivation["occurrences_skipped_few_positions"]),
        ("trips without a schedule", derivation["trips_without_schedule"]),
    ],
    columns=["accounting", "count"],
)

OTP 54.10 (window: -60s / +300s, timezone America/New_York)
on_time / early / late: 289826 / 94663 / 151267  of 535756 passages


,accounting,count
0,positions considered,2268231
1,passages derived,535756
2,REFUSED: cadence gap,3880
3,REFUSED: stop not demonstrably reached,131384
4,REFUSED: endpoint unbounded,21445
5,occurrences skipped (too few positions),1600
6,trips without a schedule,4260


## Per-route spread

Every route gets its own scoped figure. Values are exact decimals; we
convert to float here *only* to rank for display — an explicit act, fine
for exploration, never for reporting (and nothing here can become a
reported figure anyway).

In [4]:
routes = ops[(ops["metric"] == "otp") & ops["scope"].str.startswith("route:")].copy()
routes["otp_float"] = routes["value"].astype(float)
print(f"{len(routes)} route-level OTP figures")
cols = ["scope", "value", "calc_name", "calc_version", "category"]
best_and_worst = pd.concat([
    routes.sort_values("otp_float").head(5)[cols],
    routes.sort_values("otp_float").tail(5)[cols],
])
best_and_worst

172 route-level OTP figures


,scope,value,calc_name,calc_version,category
342,route:Orange,21.46,otp_v0,0.1.0,ops
173,route:Shuttle-CantonJunctionForgePark,23.81,otp_v0,0.1.0,ops
324,route:Blue,23.89,otp_v0,0.1.0,ops
343,route:Red,24.25,otp_v0,0.1.0,ops
231,route:35,30.05,otp_v0,0.1.0,ops
332,route:CR-Needham,86.03,otp_v0,0.1.0,ops
328,route:CR-Greenbush,88.11,otp_v0,0.1.0,ops
330,route:CR-Kingston,89.11,otp_v0,0.1.0,ops
325,route:CR-Fairmount,91.46,otp_v0,0.1.0,ops
331,route:CR-Lowell,93.28,otp_v0,0.1.0,ops


## Why you will never see these certified

The database itself enforces the boundary: `computed.metric_values` carries
a CHECK making `category='ops' AND certification_status='certified'`
unrepresentable, the public certified feed additionally hard-filters to
`category='ntd'`, and ops findings never block NTD certification (nor the
other way around). You are free to build dashboards, SLAs, and board
reports on these figures — Headway just guarantees they can never be
confused with what the agency reports to the FTA.